<a href="https://colab.research.google.com/github/ga426553-sudo/Classification-of-Breast-Cancer-Subtypes-machine-learning---CuMiDa-22820/blob/main/FeatureSelection_mRMR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Segundo código - Cáncer de Mama 🌸

**Feature Selection y mRMR**

### Conectar con Google Drive 🌺

In [ ]:
# ============================================
# CÓDIGO 2: mRMR INCREMENTAL (PROCESANDO POR LOTES)
# ============================================

from google.colab import drive
drive.mount('/content/drive')

import numpy as np
import pandas as pd
import pickle
from scipy import stats
from sklearn.feature_selection import f_classif
import time
from tqdm import tqdm

print("="*50)
print("CÓDIGO 2: FEATURE SELECTION INCREMENTAL")
print("="*50)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CÓDIGO 2: FEATURE SELECTION INCREMENTAL


### Carga de los datos desde Google Drive 🌺

In [ ]:
# 1. CARGAR DATOS
# ================
print(f"\n📂 Cargando datos...")

X_train = np.load('/content/drive/MyDrive/Octavo semestre/Optativa2/BreastCancer/X_train_balanced.npy')
y_train = np.load('/content/drive/MyDrive/Octavo semestre/Optativa2/BreastCancer/y_train_balanced.npy')
X_test = np.load('/content/drive/MyDrive/Octavo semestre/Optativa2/BreastCancer/X_test_clean.npy')
y_test = np.load('/content/drive/MyDrive/Octavo semestre/Optativa2/BreastCancer/y_test_clean.npy')

with open('/content/drive/MyDrive/Octavo semestre/Optativa2/BreastCancer/feature_names_filtered.pkl', 'rb') as f:
    feature_names = pickle.load(f)

print(f"✅ Datos cargados:")
print(f"   X_train: {X_train.shape} (genes: {X_train.shape[1]})")
print(f"   X_test: {X_test.shape}")
print(f"   y_train: {y_train.shape}")


📂 Cargando datos...
✅ Datos cargados:
   X_train: (206, 31215) (genes: 31215)
   X_test: (28, 31215)
   y_train: (206,)


### Desarrollo de mRMR 🌺

In [ ]:
# 2. mRMR INCREMENTAL POR LOTES
# ===============================
print(f"\n--- Aplicando mRMR incremental (procesando por lotes) ---")

def mrmr_incremental(X, y, feature_names, K=20, batch_size=1000):
    """
    Versión de mRMR que procesa los genes en lotes para no saturar la RAM
    """
    n_features = X.shape[1]
    n_samples = X.shape[0]

    print(f"📊 Configuración:")
    print(f"   Total genes: {n_features:,}")
    print(f"   Batch size: {batch_size}")
    print(f"   K (genes a seleccionar): {K}")

    # PASO 1: Calcular relevancia (F-test) para TODOS los genes, pero por lotes
    print("\n🔄 Paso 1: Calculando relevancia (F-test) por lotes...")

    relevance_scores = {}

    for start in tqdm(range(0, n_features, batch_size), desc="Procesando lotes"):
        end = min(start + batch_size, n_features)
        X_batch = X[:, start:end]

        # Calcular F-test para este lote
        f_scores, _ = f_classif(X_batch, y)

        # Guardar scores
        for i, idx in enumerate(range(start, end)):
            gene_name = feature_names[idx]
            relevance_scores[gene_name] = f_scores[i]

    # Ordenar por relevancia
    relevance_series = pd.Series(relevance_scores)
    relevance_sorted = relevance_series.sort_values(ascending=False)

    print(f"\n✅ Top 10 genes por relevancia:")
    for i, (gene, score) in enumerate(relevance_sorted.head(10).items(), 1):
        print(f"   {i:2d}. {gene}: {score:.2f}")

    # PASO 2: Selección incremental considerando redundancia
    print("\n🔄 Paso 2: Seleccionando top", K, "genes con MRMR...")

    selected = []
    not_selected = list(relevance_sorted.index)

    # El primer gen es el más relevante
    first_gene = relevance_sorted.index[0]
    selected.append(first_gene)
    not_selected.remove(first_gene)

    print(f"   Gen 1 seleccionado: {first_gene}")

    # Seleccionar los siguientes K-1 genes
    for k in range(1, K):
        print(f"   Buscando gen {k+1}/{K}...")

        best_score = -1
        best_gene = None

        # Evaluar candidatos (solo los top 1000 más relevantes para ahorrar tiempo)
        candidates = relevance_sorted.loc[not_selected].head(1000).index

        for candidate in candidates:
            # Relevancia
            rel = relevance_scores[candidate]

            # Calcular redundancia promedio con genes ya seleccionados
            redundancies = []
            for sel in selected:
                # Extraer los datos de ambos genes
                idx_candidate = feature_names.index(candidate)
                idx_sel = feature_names.index(sel)

                # Calcular correlación
                corr = np.corrcoef(X[:, idx_candidate], X[:, idx_sel])[0, 1]
                redundancies.append(abs(corr))

            avg_redundancy = np.mean(redundancies) if redundancies else 0

            # MRMR score (relevancia / redundancia)
            if avg_redundancy > 0:
                score = rel / avg_redundancy
            else:
                score = rel * 100  # Si no hay redundancia, bonus

            if score > best_score:
                best_score = score
                best_gene = candidate

        if best_gene:
            selected.append(best_gene)
            not_selected.remove(best_gene)
            print(f"      ✅ Seleccionado: {best_gene}")

    return selected

# Aplicar mRMR incremental
start_time = time.time()
selected_genes_mrmr = mrmr_incremental(
    X_train,
    y_train,
    feature_names,
    K=20,
    batch_size=1000  # Procesa de 1000 en 1000 genes
)
end_time = time.time()

print(f"\n⏱️ Tiempo total: {(end_time - start_time)/60:.2f} minutos")

print(f"\n✅ Top 20 genes seleccionados por mRMR:")
for i, gene in enumerate(selected_genes_mrmr, 1):
    print(f"{i:2d}. {gene}")


--- Aplicando mRMR incremental (procesando por lotes) ---
📊 Configuración:
   Total genes: 31,215
   Batch size: 1000
   K (genes a seleccionar): 20

🔄 Paso 1: Calculando relevancia (F-test) por lotes...


Procesando lotes: 100%|██████████| 32/32 [00:00<00:00, 171.08it/s]



✅ Top 10 genes por relevancia:
    1. NM_002644: 2535.47
    2. BM511013: 2514.49
    3. NM_182611: 2351.38
    4. THC2415390: 2106.13
    5. AY343891: 2098.78
    6. NM_000717: 2071.64
    7. NM_000493: 1918.26
    8. NM_007058: 1884.33
    9. NM_052953: 1870.20
   10. NM_148960: 1749.01

🔄 Paso 2: Seleccionando top 20 genes con MRMR...
   Gen 1 seleccionado: NM_002644
   Buscando gen 2/20...
      ✅ Seleccionado: BM511013
   Buscando gen 3/20...
      ✅ Seleccionado: NM_182611
   Buscando gen 4/20...
      ✅ Seleccionado: AY343891
   Buscando gen 5/20...
      ✅ Seleccionado: NM_000717
   Buscando gen 6/20...
      ✅ Seleccionado: THC2415390
   Buscando gen 7/20...
      ✅ Seleccionado: NM_000493
   Buscando gen 8/20...
      ✅ Seleccionado: NM_007058
   Buscando gen 9/20...
      ✅ Seleccionado: NM_052953
   Buscando gen 10/20...
      ✅ Seleccionado: NM_148960
   Buscando gen 11/20...
      ✅ Seleccionado: AK001007
   Buscando gen 12/20...
      ✅ Seleccionado: NM_000916
   Buscan

### T-test en los mejores 20 genes 🌺

In [ ]:
# 3. t-test SOBRE LOS 20 GENES
# ==============================
print(f"\n--- Aplicando t-test sobre los 20 genes ---")

# Crear DataFrames con los 20 genes
X_train_df = pd.DataFrame(X_train, columns=feature_names)
X_test_df = pd.DataFrame(X_test, columns=feature_names)

X_train_20 = X_train_df[selected_genes_mrmr]
X_test_20 = X_test_df[selected_genes_mrmr]

significant_genes = []
p_values = []

print("\n📊 Resultados t-test (p < 0.05):")
for gene in selected_genes_mrmr:
    gene_class0 = X_train_20[y_train == 0][gene].values
    gene_class1 = X_train_20[y_train == 1][gene].values

    t_stat, p_value = stats.ttest_ind(gene_class0, gene_class1, equal_var=False)
    p_values.append(p_value)

    if p_value < 0.05:
        significant_genes.append(gene)
        print(f"✅ {gene}: p = {p_value:.6f}")
    else:
        print(f"❌ {gene}: p = {p_value:.6f}")

print(f"\n📊 Genes significativos: {len(significant_genes)}")
print(f"   {significant_genes}")


--- Aplicando t-test sobre los 20 genes ---

📊 Resultados t-test (p < 0.05):
✅ NM_002644: p = 0.000000
✅ BM511013: p = 0.000000
✅ NM_182611: p = 0.000000
✅ AY343891: p = 0.000000
✅ NM_000717: p = 0.000000
✅ THC2415390: p = 0.000000
✅ NM_000493: p = 0.000000
✅ NM_007058: p = 0.000000
✅ NM_052953: p = 0.000000
✅ NM_148960: p = 0.000000
✅ AK001007: p = 0.000000
✅ NM_000916: p = 0.000000
✅ NM_201648: p = 0.000000
✅ THC2441719: p = 0.000000
✅ NM_214462: p = 0.000000
✅ NM_020918: p = 0.000000
✅ NM_207299: p = 0.000000
✅ NM_206539: p = 0.000000
✅ AF150379: p = 0.000000
✅ AW015426: p = 0.000000

📊 Genes significativos: 20
   ['NM_002644', 'BM511013', 'NM_182611', 'AY343891', 'NM_000717', 'THC2415390', 'NM_000493', 'NM_007058', 'NM_052953', 'NM_148960', 'AK001007', 'NM_000916', 'NM_201648', 'THC2441719', 'NM_214462', 'NM_020918', 'NM_207299', 'NM_206539', 'AF150379', 'AW015426']


### Subconjuntos de los datos 🌺

In [ ]:
# 4. CREAR SUBSETS (METAHEURÍSTICOS SIMULADOS)
# ==============================================
print(f"\n--- Creando subconjuntos para KNN ---")

n_sig = len(significant_genes)

# Buscar los genes del artículo por patrones comunes
mapk1_patterns = ['MAPK', '138957', 'MAPK1', 'NM_138957']
apobec_patterns = ['APOBEC', '152426', 'APOBEC3B', 'NM_152426']
enah_patterns = ['ENAH', '001008493', 'MENA', 'NM_001008493']

mapk1_candidates = [g for g in significant_genes if any(p in g for p in mapk1_patterns)]
apobec_candidates = [g for g in significant_genes if any(p in g for p in apobec_patterns)]
enah_candidates = [g for g in significant_genes if any(p in g for p in enah_patterns)]

print(f"\n🔍 Genes del artículo encontrados:")
print(f"   MAPK1 candidates: {mapk1_candidates}")
print(f"   APOBEC3B candidates: {apobec_candidates}")
print(f"   ENAH candidates: {enah_candidates}")

# Crear subsets según el artículo
subsets = {}

if len(mapk1_candidates) > 0 and len(apobec_candidates) > 0 and len(enah_candidates) > 0:
    # Caso ideal: encontramos los 3 genes del artículo
    subsets['EO'] = [mapk1_candidates[0], apobec_candidates[0], enah_candidates[0]]
    print(f"\n✨ ¡Encontrados los 3 genes del artículo!")
else:
    # Si no encontramos los genes exactos, usamos los más significativos
    if n_sig >= 3:
        subsets['EO'] = significant_genes[:3]
    elif n_sig >= 2:
        subsets['EO'] = significant_genes[:2]
    else:
        subsets['EO'] = significant_genes

# Crear los otros subsets
if n_sig >= 2:
    subsets['BBA'] = significant_genes[:2]
else:
    subsets['BBA'] = significant_genes

if n_sig >= 3:
    subsets['CSA'] = significant_genes[1:3] if n_sig >= 3 else significant_genes[:2]
else:
    subsets['CSA'] = significant_genes

subsets['RDA'] = [significant_genes[0]] if n_sig > 0 else []
subsets['GA'] = [significant_genes[-1]] if n_sig > 0 else []

print("\n📋 Subconjuntos creados:")
for name, genes in subsets.items():
    if genes:
        print(f"   {name}: {genes}")
    else:
        print(f"   {name}: (vacío)")


--- Creando subconjuntos para KNN ---

🔍 Genes del artículo encontrados:
   MAPK1 candidates: []
   APOBEC3B candidates: []
   ENAH candidates: []

📋 Subconjuntos creados:
   EO: ['NM_002644', 'BM511013', 'NM_182611']
   BBA: ['NM_002644', 'BM511013']
   CSA: ['BM511013', 'NM_182611']
   RDA: ['NM_002644']
   GA: ['AW015426']


### Guardar los resultados 🌺

In [ ]:
# 5. GUARDAR RESULTADOS
# ======================
print(f"\n--- Guardando resultados ---")

# Guardar subsets
with open('/content/drive/MyDrive/Octavo semestre/Optativa2/BreastCancer/subsets.pkl', 'wb') as f:
    pickle.dump(subsets, f)

# Guardar datos para cada subset
for name, genes in subsets.items():
    if genes:  # Solo si no está vacío
        train_data = X_train_20[genes].values
        test_data = X_test_20[genes].values

        np.save(f'/content/drive/MyDrive/Octavo semestre/Optativa2/BreastCancer/train_{name}.npy', train_data)
        np.save(f'/content/drive/MyDrive/Octavo semestre/Optativa2/BreastCancer/test_{name}.npy', test_data)
        print(f"   ✅ Guardado {name}: {train_data.shape}")

# Guardar etiquetas
np.save('/content/drive/MyDrive/Octavo semestre/Optativa2/BreastCancer/y_train_final.npy', y_train)
np.save('/content/drive/MyDrive/Octavo semestre/Optativa2/BreastCancer/y_test_final.npy', y_test)

# Guardar listas de genes
with open('/content/drive/MyDrive/Octavo semestre/Optativa2/BreastCancer/selected_genes_mrmr.pkl', 'wb') as f:
    pickle.dump(selected_genes_mrmr, f)

with open('/content/drive/MyDrive/Octavo semestre/Optativa2/BreastCancer/significant_genes.pkl', 'wb') as f:
    pickle.dump(significant_genes, f)

print("\n✅ RESULTADOS GUARDADOS EN GOOGLE DRIVE")
print("Archivos guardados en /content/drive/MyDrive/Octavo semestre/Optativa2/BreastCancer/")


--- Guardando resultados ---
   ✅ Guardado EO: (206, 3)
   ✅ Guardado BBA: (206, 2)
   ✅ Guardado CSA: (206, 2)
   ✅ Guardado RDA: (206, 1)
   ✅ Guardado GA: (206, 1)

✅ RESULTADOS GUARDADOS EN GOOGLE DRIVE
Archivos guardados en /content/drive/MyDrive/Octavo semestre/Optativa2/BreastCancer/
